# landing → bronze: 年齢各歳・男女別人口(050008_toukeika_032)

SageMaker Unified Studio の JupyterLab(Python 3 カーネル)で上から順に実行する。
データは合計 3.3MB なので Spark は使わず、boto3 だけで処理する。

1. landing の CSV を読み、文字コードを判定して UTF-8 にそろえる(BOM は外す)
2. 改行コードを LF にそろえる
3. 構造を確認する(全行の列数がヘッダと同じか / 空行・注釈行が混ざっていないか)
4. **全ファイルが確認を通ったときだけ** bronze に書き出す

| 層 | 場所 |
| --- | --- |
| landing | `s3://<bucket>/akita/landing/population_by_age/` |
| bronze | `s3://<bucket>/akita/bronze/population_by_age/` |
| 確認結果 | `s3://<bucket>/akita/_reports/population_by_age/` |

確認結果を bronze の下に置かないのは、Athena がテーブルの場所にあるファイルを
拡張子に関係なく全部データとして読むため。

出典: 「年齢別人口流動調査（県・市町村の年齢各歳、男女別人口）」(秋田県)
秋田県オープンデータカタログサイト, CC BY 4.0
https://ckan.pref.akita.lg.jp/dataset/050008_toukeika_032

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────────
import os

# True のあいだは確認だけして書き込まない。結果を見てから False にして再実行する。
DRY_RUN = True

REGION = "ap-northeast-1"
BUCKET_PREFIX = "jaws-festa-2026"
LANDING_PREFIX = "akita/landing/population_by_age/"
BRONZE_PREFIX = "akita/bronze/population_by_age/"
REPORT_PREFIX = "akita/_reports/population_by_age/"

# 1列目(年月日コード)の形。これに合わない行は注釈行の疑いとして扱う。
KEY_PATTERN = r"^\d{10}$"

In [ ]:
import boto3

session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE"), region_name=REGION)
s3 = session.client("s3")
account_id = session.client("sts").get_caller_identity()["Account"]

# アカウントリージョン名前空間のバケット名(末尾 -an は必須サフィックス)
BUCKET = f"{BUCKET_PREFIX}-{account_id}-{REGION}-an"
print("bucket:", BUCKET)

keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=LANDING_PREFIX):
    keys += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(".csv")]
keys.sort()
print(f"landing の CSV: {len(keys)} 本")
for k in keys:
    print("  ", k)
assert keys, "landing に CSV が無い"

## 1〜3. 文字コード・改行コードをそろえて構造を確認する

- 文字コード: BOM 付き UTF-8 → UTF-8 → CP932 の順に**厳格に**デコードを試す。
  どれでも読めなければそのファイルは失敗にする(化けたまま通さない)。
- 改行コード: CRLF と CR を LF に置き換える。末尾は改行1つで終わらせる。
- 構造: `csv` モジュールで1行ずつ読み、次を問題として数える。
  - 列数がヘッダと違う行
  - 空行(途中・末尾とも)
  - 1列目が年月日コード(10桁の数字)でない行 = 注釈行・出典行の疑い

ファイル間の列数の違い(1980〜2015 は128列、2020 だけ127列)は**エラーにしない**。
各ファイルの中で整合していれば bronze に通す。違いは一覧で見えるようにしておく。

In [ ]:
import csv
import io
import re

def decode(raw: bytes) -> tuple[str, str]:
    if raw.startswith(b"\xef\xbb\xbf"):
        return raw[3:].decode("utf-8"), "UTF-8 (BOMあり)"
    try:
        return raw.decode("utf-8"), "UTF-8"
    except UnicodeDecodeError:
        return raw.decode("cp932"), "CP932"  # 読めなければ例外のまま失敗させる

def newline_kind(raw: bytes) -> str:
    crlf = raw.count(b"\r\n")
    lf = raw.count(b"\n") - crlf
    cr = raw.count(b"\r") - crlf
    kinds = [n for n, c in (("CRLF", crlf), ("LF", lf), ("CR", cr)) if c]
    return "+".join(kinds) or "なし"

def normalize_and_check(key: str) -> dict:
    raw = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    result = {"key": key, "bytes_in": len(raw), "newline_in": newline_kind(raw), "problems": []}
    try:
        text, result["encoding_in"] = decode(raw)
    except UnicodeDecodeError as e:
        result["encoding_in"] = "不明"
        result["problems"].append(f"文字コードを判定できない: {e}")
        return result

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")
    if lines and lines[-1] == "":
        lines.pop()  # ファイル末尾の改行1つぶん

    rows = list(csv.reader(lines))
    header, body = rows[0], rows[1:]
    ncols = len(header)
    key_re = re.compile(KEY_PATTERN)
    for lineno, row in enumerate(body, start=2):
        if not row or all(c.strip() == "" for c in row):
            result["problems"].append(f"{lineno}行目: 空行")
        elif len(row) != ncols:
            result["problems"].append(f"{lineno}行目: 列数 {len(row)}(ヘッダは {ncols})")
        elif not key_re.match(row[0].strip()):
            result["problems"].append(f"{lineno}行目: 注釈行の疑い 1列目={row[0][:20]!r}")

    out = io.StringIO()
    csv.writer(out, lineterminator="\n").writerows(rows)
    result.update(
        columns=ncols,
        rows=len(body),
        header_tail=header[-1],
        has_age_unknown="年齢不詳" in header,
        normalized=out.getvalue().encode("utf-8"),
    )
    return result

results = [normalize_and_check(k) for k in keys]

print(f"{'ファイル':<32}{'文字コード':<16}{'改行':<6}{'列':>5}{'行':>6}  年齢不詳  問題")
for r in results:
    name = r["key"].rsplit("/", 1)[-1]
    print(f"{name:<32}{r['encoding_in']:<16}{r['newline_in']:<6}{r.get('columns', '-'):>5}{r.get('rows', '-'):>6}"
          f"  {'あり' if r.get('has_age_unknown') else 'なし':<6}  {len(r['problems'])}")
    for p in r["problems"][:10]:
        print("     -", p)
    if len(r["problems"]) > 10:
        print(f"     … ほか {len(r['problems']) - 10} 件")

failed = [r for r in results if r["problems"]]
print()
print("列数の分布:", sorted({r.get("columns") for r in results if "columns" in r}))
print("確認結果:", "OK" if not failed else f"NG {len(failed)} 本")

## 4. bronze に書き出す

全ファイルが確認を通ったときだけ書く(1本でも問題があれば何も書かない)。
出力は UTF-8(BOMなし)・LF・ファイル名は landing と同じ。

In [ ]:
import json
from datetime import datetime, timezone

if failed:
    raise SystemExit(f"確認に失敗したファイルがあるので bronze には書かない: {[r['key'] for r in failed]}")

ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "run_at": ts,
    "bucket": BUCKET,
    "source": LANDING_PREFIX,
    "target": BRONZE_PREFIX,
    "dry_run": DRY_RUN,
    "files": [{k: v for k, v in r.items() if k != "normalized"} | {"bytes_out": len(r["normalized"])} for r in results],
}

for r in results:
    out_key = BRONZE_PREFIX + r["key"].rsplit("/", 1)[-1]
    if DRY_RUN:
        print(f"[dry-run] {out_key}  {r['bytes_in']:,} -> {len(r['normalized']):,} bytes")
        continue
    s3.put_object(
        Bucket=BUCKET,
        Key=out_key,
        Body=r["normalized"],
        ContentType="text/csv; charset=utf-8",
        Metadata={"source-key": r["key"], "encoding-in": r["encoding_in"].encode("ascii", "ignore").decode() or "unknown"},
    )
    print(f"wrote s3://{BUCKET}/{out_key}  {len(r['normalized']):,} bytes")

if not DRY_RUN:
    report_key = f"{REPORT_PREFIX}{ts}.json"
    s3.put_object(Bucket=BUCKET, Key=report_key, ContentType="application/json",
                  Body=json.dumps(report, ensure_ascii=False, indent=2).encode("utf-8"))
    print(f"report: s3://{BUCKET}/{report_key}")

In [ ]:
# 書き出した bronze を読み直して、BOM・CR が残っていないことを確かめる
if not DRY_RUN:
    for r in results:
        out_key = BRONZE_PREFIX + r["key"].rsplit("/", 1)[-1]
        b = s3.get_object(Bucket=BUCKET, Key=out_key)["Body"].read()
        assert not b.startswith(b"\xef\xbb\xbf"), out_key
        assert b"\r" not in b, out_key
        assert b.decode("utf-8") == r["normalized"].decode("utf-8"), out_key
    print("bronze の読み直し OK")